# SQL — CASE WHEN, Conditional Aggregation, and PIVOT
---

<div style="padding:15px;border-left:8px solid #1a8a5a;background:#e8f5e9;border-radius:4px;">
<strong>Core Insight:</strong> Most databases don't have a native PIVOT keyword.
You build PIVOT with <code>CASE WHEN</code> inside aggregate functions — the
"conditional aggregation" pattern. It's universal, readable, and works everywhere:
DuckDB, PostgreSQL, Snowflake, BigQuery, Redshift.
</div>

### The Citi Context
Monthly capacity reports showed avg CPU per server as rows. The business team wanted
a wide-format table: one row per region, columns for each environment (dev, staging, prod).
That's a PIVOT. Built it with CASE WHEN — universally compatible, zero platform lock-in.


In [ ]:
# ── Setup: create tables in DuckDB ───────────────────────────────────────────
import duckdb
con = duckdb.connect()

con.execute("""
CREATE TABLE servers AS
SELECT * FROM (VALUES
  ('SRV-001', 'us-east', 'prod',    'gold'),
  ('SRV-002', 'us-east', 'staging', 'silver'),
  ('SRV-003', 'us-east', 'dev',     'bronze'),
  ('SRV-004', 'eu-west', 'prod',    'gold'),
  ('SRV-005', 'eu-west', 'prod',    'gold'),
  ('SRV-006', 'eu-west', 'dev',     'bronze'),
  ('SRV-007', 'ap-south','prod',   'gold'),
  ('SRV-008', 'ap-south','staging','silver')
) t(server_id, region, environment, tier)
""")

con.execute("""
CREATE TABLE daily_metrics AS
SELECT * FROM (VALUES
  ('SRV-001', DATE '2026-02-26', 85.2,  72.1),
  ('SRV-001', DATE '2026-02-25', 78.5,  68.3),
  ('SRV-002', DATE '2026-02-26', 45.0,  55.0),
  ('SRV-003', DATE '2026-02-26', 30.1,  40.2),
  ('SRV-004', DATE '2026-02-26', 91.3,  88.5),
  ('SRV-005', DATE '2026-02-26', 76.4,  80.1),
  ('SRV-006', DATE '2026-02-26', 22.0,  30.0),
  ('SRV-007', DATE '2026-02-26', 95.1,  92.3),
  ('SRV-008', DATE '2026-02-26', 55.0,  60.0)
) t(server_id, report_date, avg_cpu, avg_memory)
""")

print(con.execute("SELECT * FROM servers").df())


In [ ]:
# ── 1. PIVOT Pattern — rows to columns with CASE WHEN ────────────────────────
# Goal: one row per region, with avg CPU broken out by environment

result = con.execute("""
SELECT
    s.region,
    ROUND(AVG(CASE WHEN s.environment = 'prod'    THEN m.avg_cpu END), 1) AS prod_avg_cpu,
    ROUND(AVG(CASE WHEN s.environment = 'staging' THEN m.avg_cpu END), 1) AS staging_avg_cpu,
    ROUND(AVG(CASE WHEN s.environment = 'dev'     THEN m.avg_cpu END), 1) AS dev_avg_cpu,
    COUNT(*) AS total_servers
FROM servers s
JOIN daily_metrics m ON s.server_id = m.server_id
WHERE m.report_date = '2026-02-26'
GROUP BY s.region
ORDER BY s.region
""").df()
print("PIVOT (CASE WHEN):")
print(result.to_string())
print()

# Why AVG() not SUM()? After the CASE WHEN filter, only one non-NULL value
# per environment per group row — AVG and MAX both work equally.
# Use SUM for counts: SUM(CASE WHEN env='prod' THEN 1 ELSE 0 END)
print("Key rule: CASE WHEN inside AVG() = conditional average. Inside SUM() = conditional count.")


In [ ]:
# ── 2. Tier Distribution — COUNT and PCT with CASE WHEN ──────────────────────
result = con.execute("""
SELECT
    region,
    COUNT(*) AS total_servers,
    SUM(CASE WHEN tier = 'gold'   THEN 1 ELSE 0 END) AS gold_count,
    SUM(CASE WHEN tier = 'silver' THEN 1 ELSE 0 END) AS silver_count,
    SUM(CASE WHEN tier = 'bronze' THEN 1 ELSE 0 END) AS bronze_count,
    ROUND(100.0 * SUM(CASE WHEN tier = 'gold' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_gold
FROM servers
GROUP BY region
ORDER BY total_servers DESC
""").df()
print("Tier distribution by region:")
print(result.to_string())
print()

# Alternative: FILTER clause (DuckDB / PostgreSQL — cleaner syntax)
result2 = con.execute("""
SELECT
    region,
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE tier = 'gold')   AS gold_count,
    COUNT(*) FILTER (WHERE tier = 'silver') AS silver_count,
    COUNT(*) FILTER (WHERE tier = 'bronze') AS bronze_count
FROM servers
GROUP BY region
""").df()
print("Using FILTER clause (DuckDB/PostgreSQL equivalent):")
print(result2.to_string())


In [ ]:
# ── 3. UNPIVOT — columns to rows (reverse of PIVOT) ──────────────────────────
# Scenario: source data has avg_cpu and avg_memory as columns.
# We want: server_id, metric_type, value (long format for analysis)

result = con.execute("""
-- UNION ALL approach — works in every SQL engine
SELECT server_id, 'avg_cpu'    AS metric_type, avg_cpu    AS value FROM daily_metrics
UNION ALL
SELECT server_id, 'avg_memory' AS metric_type, avg_memory AS value FROM daily_metrics
ORDER BY server_id, metric_type
""").df()
print("UNPIVOT (wide → long format):")
print(result.head(10).to_string())
print()

# DuckDB also supports UNPIVOT syntax natively:
result2 = con.execute("""
UNPIVOT daily_metrics
ON avg_cpu, avg_memory
INTO NAME metric_type VALUE value
ORDER BY server_id
""").df()
print("Native UNPIVOT syntax (DuckDB):")
print(result2.head(8).to_string())


In [ ]:
# ── 4. Cohort Analysis Pattern ────────────────────────────────────────────────
# Group servers by first-seen date (cohort).
# Track which cohorts are still reporting data on day N after first seen.

result = con.execute("""
WITH first_report AS (
    -- Each server's cohort = the first date we saw it
    SELECT server_id, MIN(report_date) AS cohort_date
    FROM daily_metrics
    GROUP BY server_id
),
cohort_activity AS (
    SELECT
        m.server_id,
        f.cohort_date,
        m.report_date,
        (m.report_date - f.cohort_date)::INTEGER AS days_since_first
    FROM daily_metrics m
    JOIN first_report f ON m.server_id = f.server_id
)
-- PIVOT: one column per retention day
SELECT
    cohort_date,
    COUNT(DISTINCT server_id)                                          AS cohort_size,
    COUNT(DISTINCT CASE WHEN days_since_first = 0 THEN server_id END) AS day_0,
    COUNT(DISTINCT CASE WHEN days_since_first = 1 THEN server_id END) AS day_1
FROM cohort_activity
GROUP BY cohort_date
ORDER BY cohort_date
""").df()
print("Cohort analysis (retention by first-seen date):")
print(result.to_string())


## 🎤 5 Interview Q&A

**Q1: Does DuckDB have a native PIVOT keyword?**

Yes — DuckDB has `PIVOT` syntax that auto-detects values. PostgreSQL requires CASE WHEN.
Snowflake has `PIVOT`. BigQuery recently added `PIVOT`. Redshift does not.
**Always learn the CASE WHEN approach first** — it's universal and you can use it on any platform.
The CASE WHEN approach is also more readable: the column names document exactly what each value means.

---

**Q2: When would you use `FILTER (WHERE ...)` instead of `CASE WHEN`?**

`FILTER` is cleaner syntax for conditional aggregation in PostgreSQL and DuckDB:
```sql
COUNT(*) FILTER (WHERE tier = 'gold')   -- DuckDB/PostgreSQL
SUM(CASE WHEN tier = 'gold' THEN 1 ELSE 0 END)   -- Universal
```
Same result, different syntax. Use `FILTER` in DuckDB/PostgreSQL for readability.
Use `CASE WHEN` when you need cross-platform compatibility (Redshift, MySQL).

---

**Q3: What is the difference between PIVOT and UNPIVOT?**

PIVOT transforms rows → columns: multiple rows for a server become one row with multiple columns.
UNPIVOT does the reverse: multiple columns become multiple rows. Use UNPIVOT when:
- Source data is wide-format (one column per metric) but you need long-format for analysis
- You want to apply a window function over all metrics together
- You're loading into a fact table with a single `value` column

---

**Q4: What is a cohort analysis and how does CASE WHEN enable it?**

Cohort analysis groups entities by their first occurrence date, then tracks behavior over time.
CASE WHEN enables this by conditional pivoting: `COUNT(CASE WHEN days_since_first = 0 THEN id END)`.
Real use: track which batches of servers first appeared in which week, then see which cohorts
are still reporting data 7 and 30 days later. Drop in reporting = data collection issue.

---

**Q5: How would you pivot when you don't know the column values ahead of time (dynamic pivot)?**

Static pivot (what we've been doing) requires knowing values upfront — hard-coded in CASE WHEN.
Dynamic pivot: first query the distinct values, then build the SQL string programmatically.
In Python: `pd.pivot_table()` or `df.pivot()`. In SQL: dynamic SQL via stored procedures.
For an interview: state both approaches, explain why static is preferred when values are known
(readable, no injection risk, faster), and dynamic is required when values are unknown at query time.


## 📚 Key Terms

| Term | Definition |
|------|------------|
| **PIVOT** | Rotating rows into columns — group + conditional aggregation |
| **UNPIVOT** | Rotating columns into rows — UNION ALL or native UNPIVOT |
| **Conditional Aggregation** | Using CASE WHEN inside an aggregate (SUM/AVG/COUNT) |
| **FILTER clause** | PostgreSQL/DuckDB syntax for conditional aggregation — `COUNT(*) FILTER (WHERE ...)` |
| **Wide Format** | One row per entity, multiple attribute columns — good for BI display |
| **Long Format** | One row per entity+attribute pair — good for analysis, window functions |
| **Cohort Date** | The date an entity first appeared — used to group entities into cohorts |
| **Cohort Analysis** | Track behavior over time relative to cohort date |
| **Static PIVOT** | Column values known at query-write time — hard-coded in CASE WHEN |
| **Dynamic PIVOT** | Column values unknown — requires programmatic SQL generation |

## 🎯 Summary

### The Universal PIVOT Pattern
```sql
SELECT
    group_col,
    AVG(CASE WHEN category = 'A' THEN value END) AS cat_A,
    AVG(CASE WHEN category = 'B' THEN value END) AS cat_B
FROM table GROUP BY group_col;
```

### Interview Confidence Checklist
- [ ] Can write PIVOT with CASE WHEN from memory in 3 minutes
- [ ] Can explain the difference between PIVOT and UNPIVOT
- [ ] Can write cohort analysis query with CASE WHEN
- [ ] Can use FILTER clause in DuckDB/PostgreSQL
- [ ] Can name the Citi narrative: wide-format capacity report

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀
